# 🚀 Dynamic Label RemBG with Google Drive Download

> Download images from Google Drive, remove background, and generate YOLO labels.

---

## ⚠️ Prerequisites

You need a `service_account.json` file from a Google Cloud Service Account with access to the Drive folder.

```
📁 Drive (ROOT_FOLDER_ID)
├── 📁 class_1/
│   ├── img001.jpg
│   └── img002.png
├── 📁 class_2/
│   └── img003.jpg
└── ...
```

---
## 1️⃣ Setup

In [1]:
# =========================
# 1. RESET LIMPO
# =========================
!pip uninstall -y rembg onnxruntime onnxruntime-gpu numpy pillow -q

# =========================
# 2. BASE COMPATÍVEL
# =========================
!pip install numpy==2.1.3 pillow==11.0.0 -q

# =========================
# 3. ONNX GPU (versão correta atual)
# =========================
!pip install onnxruntime-gpu==1.25.0 -q

# =========================
# 4. REMBG com GPU support
# =========================
!pip install "rembg[gpu]" -q

# =========================
# 5. TORCH (GPU já no Colab)
# =========================
!pip install torch torchvision torchaudio -q

import os, warnings, re, time, hashlib, requests, threading
import numpy as np
import torch
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from PIL import Image, ImageFile, ImageDraw
from rembg import remove, new_session

ImageFile.LOAD_TRUNCATED_IMAGES = True
warnings.filterwarnings('ignore')

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 33.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
cucim-cu12 26.2.0 requires scikit-image<0.26.0,>=0.19.0, but you have scikit-image 0.26.0 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.4 which is incompatible.
gradio 5

---
## 2️⃣ Configuration

In [2]:
# ============================================================
# Configuration — Edit these values
# ============================================================

# --- Google Drive ---
ROOT_FOLDER_ID = "11_v1PDmFPnxv8N1uRj5qd6VeR6gkYAsl"  # Your Drive folder ID
SERVICE_ACCOUNT_PATH = "/content/service_account.json"  # Upload in next cell

# --- Directories ---
RAW_IMAGES_DIR = "/content/data/raw/images"  # Downloaded from Drive
OUTPUT_DIR = Path("./output")  # Final processed output
DEBUG_DIR = OUTPUT_DIR / "debug_visual"  # Debug visualizations

# --- Processing ---
IMAGE_MAX_SIZE = 640  # Max image dimension (px)
CLASS_ID = 0  # YOLO class ID
WORKERS = 4  # Parallel workers for processing

# --- 🆕 Improved rembg settings ---
ALPHA_THRESHOLD = 10  # Higher = stricter (filters metallic/semi-transparent)
MIN_OBJECT_SIZE = 100  # Min pixels to consider valid object
MODEL_NAME = "isnet-general-use"  # Higher quality than u2net

# --- JPEG compression ---
JPEG_QUALITY = 90

# --- Download settings ---
MAX_WORKERS_DL = 16  # Parallel workers for download
MAX_RETRIES = 3

# --- Deduplication ---
REMOVE_DUPLICATES = True  # Remove duplicate images by MD5

# Create output directories
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "images").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "labels").mkdir(parents=True, exist_ok=True)
DEBUG_DIR.mkdir(parents=True, exist_ok=True)

SUPPORTED_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

print(f"📁 Drive Folder ID: {ROOT_FOLDER_ID}")
print(f"📁 Raw Images: {RAW_IMAGES_DIR}")
print(f"📁 Output: {OUTPUT_DIR.absolute()}")
print(f"🖼️  Max size: {IMAGE_MAX_SIZE}px")
print(f"🏷️  Class ID: {CLASS_ID}")
print(f"🔧 Model: {MODEL_NAME}")
print(f"📏 Alpha threshold: {ALPHA_THRESHOLD}")
print(f"📏 Min object size: {MIN_OBJECT_SIZE}px")

📁 Drive Folder ID: 11_v1PDmFPnxv8N1uRj5qd6VeR6gkYAsl
📁 Raw Images: /content/data/raw/images
📁 Output: /content/output
🖼️  Max size: 640px
🏷️  Class ID: 0
🔧 Model: isnet-general-use
📏 Alpha threshold: 10
📏 Min object size: 100px


---
## 3️⃣ Upload Service Account

In [3]:
# ============================================================
# Upload service_account.json and authenticate
# ============================================================

from google.colab import files as colab_files
from google.oauth2 import service_account
from google.auth.transport.requests import Request as GoogleRequest

if not os.path.exists(SERVICE_ACCOUNT_PATH):
    print("📤 Upload your service_account.json file:")
    uploaded = colab_files.upload()
    for filename, content in uploaded.items():
        with open(SERVICE_ACCOUNT_PATH, 'wb') as f:
            f.write(content)
        print(f"✅ Saved: {filename} → {SERVICE_ACCOUNT_PATH}")
else:
    print(f"✅ service_account.json already exists")

print("\n🔑 Authenticating...")
SCOPES = ["https://www.googleapis.com/auth/drive.readonly"]
CREDENTIALS = service_account.Credentials.from_service_account_file(
    SERVICE_ACCOUNT_PATH, scopes=SCOPES
)
CREDENTIALS.refresh(GoogleRequest())
print(f"✅ Authenticated!")
print(f"   Account: {CREDENTIALS.service_account_email}")

✅ service_account.json already exists

🔑 Authenticating...
✅ Authenticated!
   Account: yolo-dock-tracker@yolo-dock-tracker.iam.gserviceaccount.com


---
## 4️⃣ Download Functions (Google Drive API)

In [4]:
# ============================================================
# Download functions from Google Drive
# ============================================================

SUPPORTED_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".gif"}

_progress_lock = threading.Lock()
_stats = {"downloaded": 0, "existing": 0, "failed": 0, "total": 0, "goal": 0}


def get_fresh_token():
    """Renew token if needed."""
    if not CREDENTIALS.valid:
        CREDENTIALS.refresh(GoogleRequest())
    return CREDENTIALS.token


def list_folder(folder_id: str) -> list:
    """List all files and subfolders in a Drive folder."""
    results = []
    page_token = None
    while True:
        params = {
            "q": f"'{folder_id}' in parents and trashed = false",
            "fields": "nextPageToken, files(id, name, mimeType)",
            "pageSize": 1000,
        }
        if page_token:
            params["pageToken"] = page_token
        resp = requests.get(
            "https://www.googleapis.com/drive/v3/files",
            params=params,
            headers={"Authorization": f"Bearer {get_fresh_token()}"},
        )
        if resp.status_code != 200:
            print(f"❌ Error listing folder: {resp.text}")
            break
        data = resp.json()
        results.extend(data.get("files", []))
        page_token = data.get("nextPageToken")
        if not page_token:
            break
    return results


def sanitize_folder_name(name: str) -> str:
    """Convert folder name to valid class name."""
    name = name.strip()
    name = re.sub(r'^\\d+\\.\\s*', '', name)
    name = re.sub(r'^peça\\s*\\d+\\s*-', '', name, flags=re.IGNORECASE)
    name = re.sub(r'[^\\w\\s-]', '', name)
    name = re.sub(r'[\\s]+', '_', name)
    name = name.lower().strip('-_')
    name = re.sub(r'_+', '_', name)
    return name[:60] if name else 'unknown'


def is_image(name: str) -> bool:
    return Path(name).suffix.lower() in SUPPORTED_EXTENSIONS


def download_file(file_id: str, output_path: Path) -> tuple:
    """Download a file from Drive."""
    if output_path.exists():
        return True, "exists"
    url = f"https://www.googleapis.com/drive/v3/files/{file_id}?alt=media"
    for attempt in range(MAX_RETRIES):
        try:
            resp = requests.get(
                url,
                headers={"Authorization": f"Bearer {get_fresh_token()}"},
                stream=True,
                timeout=30,
            )
            if resp.status_code == 429:
                time.sleep(2 ** attempt)
                continue
            if resp.status_code != 200:
                return False, f"HTTP {resp.status_code}"
            output_path.parent.mkdir(parents=True, exist_ok=True)
            with open(output_path, "wb") as f:
                for chunk in resp.iter_content(chunk_size=65536):
                    if chunk:
                        f.write(chunk)
            return True, "ok"
        except Exception as e:
            if attempt < MAX_RETRIES - 1:
                time.sleep(1)
            else:
                return False, str(e)
    return False, "max_retries"


def collect_files(folder_id: str, folder_name: str, output_dir: Path, depth: int = 0) -> list:
    """Recursively collect all files for download."""
    if depth > 3:
        return []
    files = []
    class_name = sanitize_folder_name(folder_name)
    class_dir = output_dir / class_name
    class_dir.mkdir(parents=True, exist_ok=True)
    contents = list_folder(folder_id)
    subfolders = [f for f in contents if f["mimeType"] == "application/vnd.google-apps.folder"]
    images = [f for f in contents if f["mimeType"] != "application/vnd.google-apps.folder" and is_image(f["name"])]
    for img in images:
        files.append({"file_id": img["id"], "output_path": class_dir / img["name"], "class_name": class_name})
    if depth == 1:
        for subfolder in subfolders:
            sub = list_folder(subfolder["id"])
            for img in sub:
                if img["mimeType"] != "application/vnd.google-apps.folder" and is_image(img["name"]):
                    safe = re.sub(r'[^\w.-]', '_', subfolder["name"])
                    files.append({"file_id": img["id"], "output_path": class_dir / f"{safe}_{img['name']}", "class_name": class_name})
    else:
        for subfolder in subfolders:
            files.extend(collect_files(subfolder["id"], subfolder["name"], output_dir, depth + 1))
    return files


def _download_worker(args):
    file_info = args
    success, status = download_file(file_info["file_id"], file_info["output_path"])
    with _progress_lock:
        _stats["total"] += 1
        if success:
            _stats["existing" if status == "exists" else "downloaded"] += 1
        else:
            _stats["failed"] += 1
        pct = (_stats["total"] / _stats["goal"]) * 100 if _stats["goal"] else 0
        if _stats["total"] % 100 == 0 or _stats["total"] >= _stats["goal"]:
            print(f"\r  [{_stats['total']}/{_stats['goal']}] "
                  f"downloaded: {_stats['downloaded']} | "
                  f"existing: {_stats['existing']} | "
                  f"failed: {_stats['failed']} ({pct:.1f}%)", end="")
    return success


print("✅ Download functions loaded.")

✅ Download functions loaded.


---
## 5️⃣ Download Images from Drive

In [5]:
# ============================================================
# Scan Drive folder and download images
# ============================================================

output_dir = Path(RAW_IMAGES_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

print("=" * 60)
print("📡 Scanning Drive folder...")
print(f"   ROOT_FOLDER_ID: {ROOT_FOLDER_ID}")
print("=" * 60)

root_contents = list_folder(ROOT_FOLDER_ID)
folders = [f for f in root_contents if f["mimeType"] == "application/vnd.google-apps.folder"]
root_images = [f for f in root_contents if f["mimeType"] != "application/vnd.google-apps.folder" and is_image(f.get("name", ""))]

print(f"\n📁 Folders found: {len(folders)}")
print(f"🖼️  Images at root: {len(root_images)}")

all_files = []
all_classes = set()

for i, folder in enumerate(folders):
    folder_name = folder["name"].lower()

    # Ignore truck folders
    if "caminhão" in folder_name or "caminhao" in folder_name:
        print(f"  ⏭️ Ignoring [{i+1}/{len(folders)}]: {folder['name']}")
        continue

    print(f"  Scanning [{i+1}/{len(folders)}]: {folder['name']} ...", end="")
    files = collect_files(folder["id"], folder["name"], output_dir)
    all_files.extend(files)
    all_classes.update(f["class_name"] for f in files)
    print(f" {len(files)} images")

print(f"\n📊 Total: {len(all_files)} images in {len(all_classes)} classes")
print(f"\n🔍 First classes found:")
for cls in sorted(all_classes)[:10]:
    count = sum(1 for f in all_files if f["class_name"] == cls)
    print(f"   {cls}: {count} images")
if len(all_classes) > 10:
    print(f"   ... and {len(all_classes) - 10} more classes")

# Reset stats
_stats.update({"downloaded": 0, "existing": 0, "failed": 0, "total": 0, "goal": len(all_files)})

print(f"\n🚀 Starting download with {MAX_WORKERS_DL} workers...\n")
start = time.time()

with ThreadPoolExecutor(max_workers=MAX_WORKERS_DL) as executor:
    futures = [executor.submit(_download_worker, f) for f in all_files]
    for fut in as_completed(futures):
        fut.result()

elapsed = time.time() - start
print(f"\n\n" + "=" * 60)
print(f"✅ Download completed in {elapsed:.1f}s ({len(all_files)/elapsed:.1f} files/s)")
print(f"   ✅ Downloaded: {_stats['downloaded']}")
print(f"   ♻️  Existing: {_stats['existing']}")
print(f"   ❌ Failed: {_stats['failed']}")
print(f"   📁 Destination: {RAW_IMAGES_DIR}")

📡 Scanning Drive folder...
   ROOT_FOLDER_ID: 11_v1PDmFPnxv8N1uRj5qd6VeR6gkYAsl

📁 Folders found: 5
🖼️  Images at root: 1
  ⏭️ Ignoring [1/5]: Caminhão
  Scanning [2/5]: 4. REPOSITÓRIO - 44 SKUS ... 4027 images
  Scanning [3/5]: 1. REPOSITÓRIO 1 - 1O SKUS  ... 1047 images
  Scanning [4/5]: 3. REPOSITÓRIO 3 - 50 SKUS  ... 4237 images
  Scanning [5/5]: 2. REPOSITÓRIO 2 - 50 SKUS  ... 4262 images

📊 Total: 13573 images in 1 classes

🔍 First classes found:
   unknown: 13573 images

🚀 Starting download with 16 workers...

  [13573/13573] downloaded: 8 | existing: 13565 | failed: 0 (100.0%)

✅ Download completed in 1.7s (8157.5 files/s)
   ✅ Downloaded: 8
   ♻️  Existing: 13565
   ❌ Failed: 0
   📁 Destination: /content/data/raw/images


---
## 6️⃣ Remove Duplicates (MD5)

In [6]:
# ============================================================
# Remove duplicate images by MD5
# ============================================================

if not REMOVE_DUPLICATES:
    print("⏭️ Skipping deduplication (REMOVE_DUPLICATES=False)")
else:
    raw_dir = Path(RAW_IMAGES_DIR)

    print("🔍 Searching for duplicates...")

    all_raw_images = []
    for ext in SUPPORTED_EXTS:
        all_raw_images.extend(raw_dir.rglob(f"*{ext}"))
        all_raw_images.extend(raw_dir.rglob(f"*{ext.upper()}"))

    print(f"📊 Total images found: {len(all_raw_images)}")

    def md5_file(path):
        h = hashlib.md5()
        try:
            with open(path, 'rb') as f:
                while chunk := f.read(65536):
                    h.update(chunk)
            return h.hexdigest()
        except Exception:
            return None

    hash_map = {}
    duplicates = []

    print(f"⚙️  Calculating MD5 hashes...")

    with ThreadPoolExecutor(max_workers=MAX_WORKERS_DL) as executor:
        futures_map = {executor.submit(md5_file, p): p for p in all_raw_images}
        for future in as_completed(futures_map):
            p = futures_map[future]
            h = future.result()
            if h is None:
                continue
            if h in hash_map:
                duplicates.append(p)
            else:
                hash_map[h] = p

    print(f"\n📊 Deduplication result:")
    print(f"   Total images: {len(all_raw_images)}")
    print(f"   🔁 Duplicates: {len(duplicates)}")
    print(f"   ✅ Unique: {len(all_raw_images) - len(duplicates)}")

    if duplicates:
        print(f"\n🗑️  Removing {len(duplicates)} duplicates...")
        for dup_path in duplicates:
            try:
                dup_path.unlink()
            except Exception as e:
                print(f"   ⚠️ Cannot remove {dup_path.name}: {e}")
        print(f"   ✅ {len(duplicates)} duplicates removed!")

🔍 Searching for duplicates...
📊 Total images found: 13570
⚙️  Calculating MD5 hashes...


KeyboardInterrupt: 

---
## 7️⃣ Create rembg Session

In [7]:
# ============================================================
# Create rembg session with auto GPU detection
# ============================================================

providers = ["CUDAExecutionProvider"] if torch.cuda.is_available() else []
session = new_session(model_name=MODEL_NAME, providers=providers)

print(f"✅ Session created")
print(f"   Model: {MODEL_NAME}")
print(f"   Providers: {providers if providers else ['CPU']}")

✅ Session created
   Model: isnet-general-use
   Providers: ['CUDAExecutionProvider']


---
## 8️⃣ Process Images

In [ ]:
# ============================================================
# 🚀 REMBG GPU PIPELINE (CORRIGIDO + ESTÁVEL)
# ============================================================

import os
import numpy as np
import torch
from pathlib import Path
from PIL import Image, ImageDraw
from concurrent.futures import ThreadPoolExecutor, as_completed

from rembg import remove, new_session

# ============================================================
# CONFIG
# ============================================================

MODEL_NAME = "isnet-general-use"

session = new_session(
    model_name=MODEL_NAME,
    providers=["CUDAExecutionProvider"]  # 🔥 força GPU
)

print("✅ Session ready")

# ============================================================
# PATHS
# ============================================================

raw_dir = Path(RAW_IMAGES_DIR)
files = list(raw_dir.rglob("*.jpg")) + list(raw_dir.rglob("*.png"))
files = list(set(files))

print(f"🚀 Images: {len(files)}")

# ============================================================
# OUTPUT
# ============================================================

os.makedirs(OUTPUT_DIR / "images", exist_ok=True)
os.makedirs(OUTPUT_DIR / "labels", exist_ok=True)
os.makedirs(DEBUG_DIR, exist_ok=True)

# ============================================================
# PARAMS
# ============================================================

ALPHA_THRESHOLD = 10
IMAGE_MAX_SIZE = 1024

# ============================================================
# BBOX
# ============================================================

def get_bbox(alpha, w, h):
    pos = np.where(alpha > ALPHA_THRESHOLD)

    if pos[0].size == 0:
        return None

    ymin, ymax = np.min(pos[0]), np.max(pos[0])
    xmin, xmax = np.min(pos[1]), np.max(pos[1])

    bw = (xmax - xmin) / w
    bh = (ymax - ymin) / h
    xc = (xmin + (xmax - xmin) / 2) / w
    yc = (ymin + (ymax - ymin) / 2) / h

    if bw <= 0 or bh <= 0:
        return None

    return xc, yc, bw, bh, xmin, ymin, xmax, ymax

# ============================================================
# PROCESS SINGLE IMAGE (🔥 GPU SAFE)
# ============================================================

def process_file(file_path):
    try:
        img = Image.open(file_path).convert("RGB")

        # resize
        if max(img.size) > IMAGE_MAX_SIZE:
            ratio = IMAGE_MAX_SIZE / max(img.size)
            img = img.resize(
                (int(img.width * ratio), int(img.height * ratio)),
                Image.BILINEAR
            )

        # 🔥 GPU inference (SINGLE IMAGE ONLY)
        output = remove(img, session=session)

        arr = np.array(output)

        if arr.shape[2] < 4:
            return False

        alpha = arr[:, :, 3]

        bbox = get_bbox(alpha, img.width, img.height)

        if bbox is None:
            return False

        xc, yc, bw, bh, xmin, ymin, xmax, ymax = bbox

        # class id (IMPORTANTE: já deve existir no seu pipeline)
        cname = file_path.parent.name.lower()
        cid = class_map.get(cname, 0)

        label = f"{cid} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}"

        name = file_path.stem

        # save image
        img.save(OUTPUT_DIR / "images" / f"{name}.jpg", "JPEG", quality=90)

        # save label
        (OUTPUT_DIR / "labels" / f"{name}.txt").write_text(label)

        # debug
        debug = img.copy()
        draw = ImageDraw.Draw(debug)
        draw.rectangle([xmin, ymin, xmax, ymax], outline="red", width=3)
        debug.save(DEBUG_DIR / f"bbox_{name}.jpg")

        return True

    except Exception as e:
        print(f"❌ {file_path.name}: {e}")
        return False

# ============================================================
# EXECUTION (PARALELO IO + GPU SAFE)
# ============================================================

print("🚀 Starting processing...")

WORKERS = 6  # NÃO aumenta muito (GPU é gargalo aqui)

with ThreadPoolExecutor(max_workers=WORKERS) as executor:
    results = list(executor.map(process_file, files))

success = sum(results)

# ============================================================
# RESULT
# ============================================================

print("\n✅ DONE")
print(f"📦 Total: {len(files)}")
print(f"✔ Success: {success}")
print(f"⚠️ Failed: {len(files) - success}")

✅ Session ready
🚀 Images: 13545
🚀 Starting processing...
❌ 20250826_145509.jpg: name 'class_map' is not defined
❌ IMG_20250725_111404.jpg: name 'class_map' is not defined
❌ IMG_20250710_133951.jpg: name 'class_map' is not defined
❌ IMG_20250717_163427.jpg: name 'class_map' is not defined
❌ IMG_20250801_151556.jpg: name 'class_map' is not defined
❌ IMG_20250717_160913.jpg: name 'class_map' is not defined
❌ IMG_20250709_170951.jpg: name 'class_map' is not defined
❌ IMG_20250717_170119.jpg: name 'class_map' is not defined
❌ IMG_20250730_141228.jpg: name 'class_map' is not defined
❌ IMG_20250717_133025.jpg: name 'class_map' is not defined
❌ IMG_20250730_160233.jpg: name 'class_map' is not defined
❌ IMG_20250710_111615.jpg: name 'class_map' is not defined
❌ IMG_20250725_162042.jpg: name 'class_map' is not defined


---
## 9️⃣ Generate dataset.yaml and classes.json

In [ ]:
# ============================================================
# Generate dataset.yaml and classes.json
# ============================================================

import yaml
import json

# Scan classes from folder structure
raw_dir = Path(RAW_IMAGES_DIR)
classes = set()
for folder in raw_dir.iterdir():
    if folder.is_dir():
        classes.add(folder.name)

classes = sorted(classes)
nc = len(classes)

print(f"📊 Found {nc} classes")
print(f"   First: {classes[:5]}")
if nc > 5:
    print(f"   ... Last: {classes[-3:]}")

# Create class mapping
class_mapping = {name: idx for idx, name in enumerate(classes)}

# dataset.yaml
YAML_PATH = OUTPUT_DIR / "dataset.yaml"
JSON_PATH = OUTPUT_DIR / "classes.json"

dataset_cfg = {
    "path": str(OUTPUT_DIR),
    "train": "images/train",
    "val": "images/val",
    "nc": nc,
    "names": classes,
}

with open(YAML_PATH, "w", encoding="utf-8") as f:
    f.write("# YOLOv8 Dataset\n")
    f.write("# Generated by Dynamic Label RemBG\n")
    f.write(f"# Classes: {nc}\n\n")
    yaml.dump(dataset_cfg, f, default_flow_style=False, allow_unicode=True)

print(f"\n✅ dataset.yaml: {YAML_PATH}")

# classes.json
with open(JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(class_mapping, f, ensure_ascii=False, indent=2)

print(f"✅ classes.json: {JSON_PATH}")

print(f"\n📋 dataset.yaml preview:")
with open(YAML_PATH) as f:
    for line in f:
        if line.startswith("names:"):
            print(f"   {line.strip()}  ({nc} total)")
        elif not line.startswith("#"):
            print(f"   {line.strip()}")

---
## 🔟 Create YOLO Dataset Structure (Train/Val Split)

In [ ]:
# ============================================================
# Create YOLO dataset structure with train/val split
# ============================================================

import shutil
import random

VAL_SPLIT = 0.2  # 20% for validation
random.seed(42)

# Create directories
for split in ["train", "val"]:
    (OUTPUT_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (OUTPUT_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

print(f"📁 Created YOLO structure:")
print(f"   {OUTPUT_DIR / 'images/train'}")
print(f"   {OUTPUT_DIR / 'images/val'}")
print(f"   {OUTPUT_DIR / 'labels/train'}")
print(f"   {OUTPUT_DIR / 'labels/val'}")

# Group images by class (folder)
raw_dir = Path(RAW_IMAGES_DIR)
class_images = {}

for folder in raw_dir.iterdir():
    if folder.is_dir():
        class_name = folder.name
        images = []
        for ext in SUPPORTED_EXTS:
            images.extend(folder.glob(f"*{ext}"))
            images.extend(folder.glob(f"*{ext.upper()}"))
        if images:
            class_images[class_name] = list(set(images))

print(f"\n📊 Processing {len(class_images)} classes...")

# Stratified split
train_count = 0
val_count = 0

for class_name, images in class_images.items():
    random.shuffle(images)
    split_idx = int(len(images) * (1 - VAL_SPLIT))

    # Ensure at least 1 in each split
    if len(images) > 1:
        split_idx = max(1, min(split_idx, len(images) - 1))

    train_imgs = images[:split_idx]
    val_imgs = images[split_idx:]

    # Copy to train
    for img in train_imgs:
        dest_img = OUTPUT_DIR / "images" / "train" / f"{class_name}_{img.name}"
        dest_lbl = OUTPUT_DIR / "labels" / "train" / f"{class_name}_{img.stem}.txt"

        # Check if label exists in original location
        orig_lbl = img.with_suffix(".txt")
        if orig_lbl.exists():
            shutil.copy2(img, dest_img)
            shutil.copy2(orig_lbl, dest_lbl)
            train_count += 1

    # Copy to val
    for img in val_imgs:
        dest_img = OUTPUT_DIR / "images" / "val" / f"{class_name}_{img.name}"
        dest_lbl = OUTPUT_DIR / "labels" / "val" / f"{class_name}_{img.stem}.txt"

        orig_lbl = img.with_suffix(".txt")
        if orig_lbl.exists():
            shutil.copy2(img, dest_img)
            shutil.copy2(orig_lbl, dest_lbl)
            val_count += 1

total = train_count + val_count
print(f"\n✅ Split complete:")
print(f"   🟢 Train: {train_count} ({train_count/total*100:.1f}%)")
print(f"   🔵 Val:   {val_count} ({val_count/total*100:.1f}%)")
print(f"   Total: {total}")

---
## 1️⃣1️⃣ Verify Output

In [ ]:
# ============================================================
# Verify output
# ============================================================

out_images = list((OUTPUT_DIR / "images").glob("*.jpg"))
out_labels = list((OUTPUT_DIR / "labels").glob("*.txt"))
debug_images = list(DEBUG_DIR.glob("*.jpg"))

print(f"📁 Output structure:")
print(f"   images/: {len(out_images)}")
print(f"   labels/: {len(out_labels)}")
print(f"   debug_visual/: {len(debug_images)}")

if out_labels:
    print(f"\n📋 Sample label:")
    sample_label = out_labels[0].read_text().strip()
    print(f"   {out_labels[0].name}: {sample_label}")

total_size = sum(f.stat().st_size for f in out_images) / 1e6
print(f"\n💾 Total size: {total_size:.1f} MB")